<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color= '#BFD72F'>**Methodology** </font><a class="anchor" id='top'></a>

- [1. Abstract](#1)
- [2. Import Libraries](#2)
- [3. Metadata](#3)
- [4. Import Datasets](#4)
- [5. Feature Selection](#5)
    - [5.1 Filter Methods](#5_1)
        - [5.1.1 Removing Constant Features](#5_1_1)
        - [5.1.2 Correlation between features - Redundant Features](#5_1_2)
        - [5.1.3 Correlation with the target - Relevant Features](#5_1_3)
            - [5.1.3.1 Spearman Correlation](#5_1_3_1)
    - [5.2 Wrapped Methods](#5_2)
        - [5.2.1 RFE](#5_2_1)
    - [5.3 Embedded Methods](#5_3)
        - [5.3.1 Lasso](#5_3_1)
    - [5.4 Comparison between Models](#5_4)
- [6 Save the Data](#6)
- [7 End of The Notebook](#7)


<a class="anchor" id="1">

# **1. Abstract**

[Back to TOP](#TOP)
</a>

This project aims to develop a predictive model capable of estimating car prices based on their characteristics, using the Cars 4 You dataset. The goal is to support the company’s evaluation process by introducing an automated solution that accelerates car assessments and improves overall efficiency.

The current phase of the project focuses on feature selection, aiming to identify the most relevant features for predicting car prices. Different techniques were applied to evaluate the importance and contribution of each feature, reduce redundancy, and minimize noise in the data and improve signal.

In this notebook, many of the sections include helper functions created to streamline the implementation of feature selection techniques. These functions make it easier to apply the same process consistently across the training, test, and validation datasets.

<a class="anchor" id="2">

# **2. Import Libraries**

[Back to TOP](#TOP)
</a>

The following libraries will help us develop the analyses and model for this project. Pandas and Numpy, provide the efficient tools for data manipulation, cleaning and numerical computations. Matplotlib and Seaborn are used to create clear and informative visualizations. Scikit-learn offers a range of Machine Learning tools for model training, spliting the data and evaluate model performance. Finally, os, math and ceil are imported to support file management and mathematical operations.

In [63]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math


#filter methods
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import spearmanr
# spearman 
from sklearn.feature_selection import SelectKBest, f_regression

# mutual information
from sklearn.feature_selection import mutual_info_classif

#wrapper methods
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE


# embedded methods
from sklearn.linear_model import Lasso

# Load Created Functions
from feature_selection import *

# Set random seed for reproducibility
np.random.seed(40111) 


<a class="anchor" id="3">

# **3. Create Meta Data**

[Back to TOP](#TOP)
</a>

Understanding the features helps interpret the data correctly and supports subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



<a class="anchor" id="4">

# **4. Import Dataset**

[Back to TOP](#TOP)
</a>

In this section, we load the preprocessed datasets that were previously saved after the data preprocessing steps. These datasets are ready for feature selection and further analysis.

In [64]:
# Define relative path to the preprocessed data folder (outside "notebooks/")
data_path = "../data_preprocessed/"

# Load preprocessed datasets
X_train = pd.read_csv(f"{data_path}X_train_preprocessed.csv", index_col=0)
X_val   = pd.read_csv(f"{data_path}X_val_preprocessed.csv", index_col=0)
test    = pd.read_csv(f"{data_path}test_preprocessed.csv", index_col=0)

y_train = pd.read_csv(f"{data_path}y_train.csv", index_col=0).squeeze()  # convert DataFrame → Series
y_val   = pd.read_csv(f"{data_path}y_val.csv", index_col=0).squeeze()

In [65]:
X_train.shape

(60778, 234)

<a class="anchor" id="5">

# **5. Feature Selection**

[Back to TOP](#TOP)
</a>

In this stage of the project, we design and implement a structured feature selection pipeline. The process begins by removing the feature 'paintQuality%' derived from the mechanic’s evaluation, as these feature will not be considered for modelling.

After this initial reduction, we apply a sequence of filter-based methods to identify and exclude features that are constant, redundant, or highly correlated with the target variable.

Next, we incorporate a wrapper method which is the Recursive Feature Elimination (RFE) to evaluate subsets of features based on model performance. Finally, we use Lasso regression as an embedded method, allowing the model itself to perform feature selection through regularisation.

The rationale is to apply these three complementary approaches (Filter, Wrapper, and Embedded) to the same preprocessed dataset to obtain a robust and diversified assessment of feature relevance. At the end of the process, we compare the sets of features retained by each method. The final subset of features used for modelling is defined by selecting those chosen by at least a predefined number of methods (e.g., 1, 2, or all 3), a threshold that can be adjusted according to project needs.
This ensures that the final modelling dataset is composed only of the most informative and consistently relevant features.

We also want to note that throughout this notebook we repeatedly inspect the shape of the X_train dataset to verify whether features are being retained or removed as intended during each step of the selection process.


<a class="anchor" id="5_1">

## **5.1 Remove paintQuality%**

[Back to TOP](#TOP)
</a>

This decision is supported by our Exploratory Data Analysis (EDA), where we concluded that these attribute originate from the mechanic’s evaluation. Since the objective of this machine learning project is precisely to replace this stage of the process, the model must operate without relying on such information.

In [66]:
X_train.shape

(60778, 234)

In [67]:
X_train = X_train.drop(['paintQuality%', 'hasDamage'], axis=1, errors='ignore')
X_val = X_val.drop(['paintQuality%', 'hasDamage'], axis=1, errors='ignore')

In [68]:
X_train.shape

(60778, 232)

We create a copy of X_train and assign it to a dedicated variable used exclusively for fitting the transformers. The purpose of defining this variable at the start is to preserve the original set of features, ensuring subsequent transformations are consistently learned from the initial training data.

In [69]:
# Save a copy of X_train to use for fitting transformers only on training data with the initial features
X_train_fit_init = X_train.copy()

<a class="anchor" id="5_1">

## **5.1 Filter Methods**

[Back to TOP](#TOP)
</a>

Filter methods are ideal for an initial screening of high-dimensional datasets due to their simplicity, even though they are considered less effective than model-based methods.
In this section, we will explore three filter-based techniques to select the features.

<a class="anchor" id="5_1_1">

### **5.1.1 Removing Constant Features**

[Back to TOP](#TOP)
</a>

This method identifies features that have low or zero variance, which are unlikely to contribute meaningful information to the model, and thus can help to reduce noise and improve efficiency.

In [70]:
# Used to fit transformers only on training data
X_train_fit = X_train.copy()

X_train = apply_variance_filter(X_train_fit, X_train, threshold=0.001, return_summary=True)
X_val = apply_variance_filter(X_train_fit, X_val, threshold=0.001, return_summary=False)

Total features kept: 130
Features selected: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'car_age', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Ford', 'ohe_Brand_Hyundai', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel', 'ohe_Brand_Skoda', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A7', 'ohe_model_ A8', 'ohe_model_ Adam', 'ohe_model_ Amarok', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Auris', 'ohe_model_ Avensis', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ C-MAX', 'oh

In [71]:
#Check the amount of features that were removed
X_train.shape

(60778, 130)

<a class="anchor" id="5_1_2">

### **5.1.2 Correlation between features - Redundant Features**

[Back to TOP](#TOP)
</a>

We use correlation measures to identify highly correlated features. Removing redundant features helps reduce multicollinearity and simplifies the dataset, making the model more efficient and easier to interpret.


In [72]:
# Create a copy of X_train to fit the following method
X_train_fit = X_train.copy()

X_train = remove_highly_correlated_features(X_train_fit, X_train, threshold=0.95, return_summary=True)
X_val = remove_highly_correlated_features(X_train_fit, X_val, threshold=0.95, return_summary=False)

Total features kept: 126
Features selected: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Ford', 'ohe_Brand_Hyundai', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel', 'ohe_Brand_Skoda', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A7', 'ohe_model_ A8', 'ohe_model_ Adam', 'ohe_model_ Amarok', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Auris', 'ohe_model_ Avensis', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ C-MAX', 'ohe_model_ CL

In [73]:
# Check that some features were removed
X_train.shape

(60778, 126)

<a class="anchor" id="5_1_3">

### **5.1.3 Correlation with the target - Relevant Features**

[Back to TOP](#TOP)
</a>

These processes consist of setting a correlation threshold and keeping only the features that show a meaningful relationship with the target feature. 

We decided to utilize Spearman correlation because it measures monotonic relationships rather than being strictly limited to linear relationships, as is the case with the Pearson coefficient. This is crucial because, in the context of car pricing, the depreciation (for example mileage) or the impact of other characteristics is often non-linear. Additionally, Spearman is based on rankings, which makes it significantly more robust to outliers, providing a more reliable measure of feature relevance for our dataset.

In [74]:
# Create a copy of X_train to fit the following methods
X_train_fit = X_train.copy()

X_train = spearman_correlation_selection(X_train_fit, X_train, y_train, threshold=0.2, return_summary=True)
X_val = spearman_correlation_selection(X_train_fit, X_val, y_train, threshold=0.2, return_summary=False)

Total features kept: 22
Features selected by Spearman method: ['year', 'is_automatic', 'fuel_efficiency_score', 'engineSize', 'brand_avg_engineSize', 'mileage', 'ohe_mileage_category_Very Low', 'is_recent_car', 'ohe_transmission_Semi-Auto', 'tax_efficiency', 'brand_avg_age', 'mpg', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel', 'tax', 'ohe_mileage_category_Medium', 'fueltype_avg_mpg', 'ohe_model_ C Class', 'tax_to_engine_ratio', 'ohe_Brand_Ford', 'ohe_model_ Corsa', 'ohe_model_ Fiesta']
Nº Features eliminated: 104


In [75]:
# Save the list of selected features with Filter Methods to use later for comparison
filter_cols = X_train.columns.tolist()

<a class="anchor" id="5_2">

## **5.2 Wrapped Methods**

[Back to TOP](#TOP)
</a>

Wrapper methods evaluate subsets of features by training and assessing a predictive model, making them more computationally expensive but generally more effective than filter approaches. Because they consider feature interactions and model performance directly, they provide a more nuanced selection.
In this section, we apply a wrapper-based technique to identify the most informative features for our target variable.

<a class="anchor" id="5_2_1">

### **5.2.1 RFE**

[Back to TOP](#TOP)
</a>

Here we use LinearRegression as the estimator for RFE because it is simple, fast to train, and provides a clear interpretation of feature importance through its coefficients.
We selected 30 features because this represents a meaningful number of predictors while still reducing the dimensionality of the dataset. This choice offers a good balance between model simplicity and information retention, and it also allows for a consistent comparison with the other feature selection methods. We also observed that the filter methods retained fewer than 30 features, which further supports using this threshold for RFE.

In [76]:
wrapped_cols = rfe_selection(X_train_fit_init, X_train, y_train, LinearRegression(), n_features=30, return_summary=True)

Total features kept: 30
Features selected by RFE method: ['ohe_model_ 7 Series', 'ohe_model_ 8 Series', 'ohe_model_ A8', 'ohe_model_ California', 'ohe_model_ Caravelle', 'ohe_model_ G Class', 'ohe_model_ GLE Clas', 'ohe_model_ GLS Clas', 'ohe_model_ Land Cruiser', 'ohe_model_ M2', 'ohe_model_ M3', 'ohe_model_ M4', 'ohe_model_ M5', 'ohe_model_ PROACE VERSO', 'ohe_model_ Q5', 'ohe_model_ Q7', 'ohe_model_ Q8', 'ohe_model_ R8', 'ohe_model_ RS6', 'ohe_model_ S8', 'ohe_model_ SL CLASS', 'ohe_model_ Supra', 'ohe_model_ Touareg', 'ohe_model_ V Class', 'ohe_model_ X4', 'ohe_model_ X5', 'ohe_model_ X6', 'ohe_model_ X7', 'ohe_model_GLE Class', 'ohe_model_Tigua']
Nº Features eliminated: 202


<a class="anchor" id="5_3">

## **5.3 Embedded Methods**

[Back to TOP](#TOP)
</a>

Embedded methods perform feature selection as part of the model training process itself. By incorporating regularisation or other model-driven selection criteria, they offer a balance between the efficiency of filter methods and the predictive power of wrapper methods.
Now, we use an embedded method to select features based on their contribution to the model’s optimisation procedure.

<a class="anchor" id="5_3_1">

### **5.3.1 Lasso**

[Back to TOP](#TOP)
</a>

Lasso is a linear regression method that incorporates L1 regularisation to constrain the model’s coefficients. This penalty forces some coefficients to shrink exactly to zero, which means that Lasso performs feature selection as part of the training process. By using this method, we can also define a threshold that helps determine which features should be considered irrelevant when their coefficients are close to zero.

In [ ]:
embedded_cols = lasso_selection(X_train_fit_init, X_train, y_train, threshold=0.1, return_summary=True)

Total features kept: 131
Features selected by Lasso method: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'car_age', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Ford', 'ohe_Brand_Hyundai', 'ohe_Brand_Opel', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ 7 Series', 'ohe_model_ 8 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A8', 'ohe_model_ Adam', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ CLS Class', 'ohe_model_ Caddy Maxi Life', 'ohe_model_ California', 'ohe_model_ Caravelle',

<a class="anchor" id="5_4">

## **5.4 Comparison between Models**

[Back to TOP](#TOP)
</a>

As mentioned previously, we will compare the outputs of the different feature selection methods and identify the features that consistently appear as important for our regression model.
The function we defined performs this comparison by selecting the features that are retained by at least a minimum number of methods (n_agreed). For example, if n_agreed = 2, the function returns all features that were selected by two or more of the three methods.

We assign the resulting set of selected features to a variable so that we can later use it to build our final modelling dataset, which will include only these chosen predictors.

In [77]:
selected_features = comparison_feature_selection(X_train, filter_cols, wrapped_cols, embedded_cols, n_agreed=2, return_summary=True)

Total features before selection: 22
Total features selected by at least 2 methods: 20
Selected features: ['year', 'is_automatic', 'fuel_efficiency_score', 'engineSize', 'brand_avg_engineSize', 'mileage', 'ohe_mileage_category_Very Low', 'is_recent_car', 'ohe_transmission_Semi-Auto', 'tax_efficiency', 'brand_avg_age', 'mpg', 'ohe_Brand_Opel', 'tax', 'ohe_mileage_category_Medium', 'ohe_model_ C Class', 'tax_to_engine_ratio', 'ohe_Brand_Ford', 'ohe_model_ Corsa', 'ohe_model_ Fiesta']


In [ ]:
# Define final datasets with selected features
X_train_final = X_train[selected_features]
X_val_final = X_val[selected_features]
test_final = test[selected_features]

<a class="anchor" id="6">

# **6 Save the Data**

[Back to TOP](#TOP)
</a>

This section saves the processed dataset containing only the selected features, ensuring it is ready to be used in the model assessment notebook.

In [61]:
# Go one level up from the notebooks folder to reach the repo root
selected_dir = "../data_feature_selected"
os.makedirs(selected_dir, exist_ok=True)

# Save processed datasets as CSV files
X_train_final.to_csv(f"{selected_dir}/X_train_final.csv", index=True)
X_val_final.to_csv(f"{selected_dir}/X_val_final.csv", index=True)
test_final.to_csv(f"{selected_dir}/test_final.csv", index=True)

y_train.to_csv(f"{selected_dir}/y_train.csv", index=True)
y_val.to_csv(f"{selected_dir}/y_val.csv", index=True)

<a class="anchor" id="7">

# **7 End of the Notebook**

[Back to TOP](#TOP)
</a>

The feature selection process refined the dataset and enhancing the overall performance of the predictive model. Several complementary techniques were applied to identify the most relevant features, reduce redundancy, and remove irrelevant features. This systematic approach helped ensure that the final set of features captured the most meaningful relationships with the target feature, improving both model efficiency and interpretability. Additionally, the dataset’s shape was continuously monitored after each step to confirm that all transformations were correctly applied and that the data remained consistent throughout the process.